# Loading Structured Data from CSV Files with LangChain

In advanced Retrieval-Augmented Generation (RAG) systems, the knowledge base rarely consists solely of unstructured text documents (like PDFs or articles). Real-world enterprise data often resides in structured formats such as CSVs, databases, or JSON files. This notebook introduces the critical skill of ingesting this structured data into a format that LangChain can effectively process: the `Document` object.

The core challenge when loading structured data is not just reading the file, but intelligently mapping its columns to the appropriate parts of the document structure. We must decide which column contains the primary narrative content (the `page_content`), and which columns contain valuable contextual information that should be preserved as searchable metadata (e.g., dates, IDs, source names). By mastering specialized loaders like `CSVLoader`, developers ensure that every piece of context—from a website URL to an employee count—is captured and available for the retrieval step.

This capability is foundational for building robust LangGraph agents. When an agent needs to answer a question based on structured knowledge (e.g., "What was the founding year and industry of Company X?"), it relies entirely on the metadata richness provided by these loaders. By understanding how to load, partition, and enrich data from CSVs, you gain the ability to build highly accurate, context-aware RAG pipelines that move beyond simple text matching into sophisticated knowledge graph querying.

### Learning Objectives

Upon completing this notebook, you will be able to:
*   Utilize `CSVLoader` to ingest structured data from a local CSV file path.
*   Differentiate between primary content columns (`content_columns`) and contextual metadata columns (`metadata_columns`).
*   Understand how the loader maps specific CSV columns into the `page_content` and `metadata` attributes of LangChain's `Document` object.
*   Prepare structured data for use as a knowledge source within an advanced RAG pipeline or LangGraph workflow.


In [3]:
# from langchain_docling.loader import DoclingLoader

# FILE_PATH = "https://arxiv.org/pdf/2408.09869"

# loader = DoclingLoader(file_path=FILE_PATH)

# # Load all documents
# documents = loader.load()

# # For large datasets, lazily load documents
# for document in loader.lazy_load():
#     print(document)

In [4]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from pathlib import Path
from pprint import pp

/var/folders/vw/pv_ljttx1xj4wvvkd3gch0dr0000gn/T/ipykernel_88564/2293542744.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.csv_loader import CSVLoader


### File Path Definition and Validation

This cell uses the `pathlib.Path` object to define a platform-independent path to the CSV knowledge source (`organizations.csv`). The subsequent call to `.exists()` is crucial for validating that the file is present at the specified location before attempting any loading or processing, preventing runtime errors.


In [5]:
# define the path to csv file

# Use pathlib.Path to create an object representing the file system path.
file_path = Path("../knowledge-source/organizations.csv")

# check if path exists
# The .exists() method returns a boolean indicating if the file or directory exists at this path.
print(file_path.exists())


True


### CSV Loading and Metadata Extraction

This cell initializes the `CSVLoader` from LangChain, which is responsible for loading data from a specified CSV file. It intelligently structures the loaded documents by designating specific columns: `source_column` (for industry context), `metadata_columns` (for structured details like website or founding date), and `content_columns` (for the main text content).

This process is crucial for ensuring that when we index the data, the resulting vector embeddings are rich with both textual content and valuable contextual metadata.


In [6]:
# create the csv loader

# Initialize the CSVLoader object.
loader = CSVLoader(file_path=file_path,
                   source_column="Industry", # Specifies which column provides the primary source/context for the document.
                   metadata_columns=["Website", "Founded", "Number of employees"], # Lists columns whose data should be extracted and stored as metadata.
                   content_columns=["Description"])
# The 'loader' object is now configured to load and parse the CSV file according to these column specifications.


### Document Loading

This cell executes the document loading process. The `loader` object, which was previously initialized (e.g., using a CSV loader), is called with `.load()` to read all documents from the source and store them in the `documents` variable. This step transforms raw data into structured LangChain/LlamaIndex Document objects ready for embedding and retrieval.


In [7]:
# load the documents

# Call the load method on the initialized loader object.
# This reads all documents from the source (e.g., CSV file) 
# and stores them as a list of LangChain/LlamaIndex Document objects in 'documents'.
documents = loader.load()


### Code Explanation

This cell simply calculates and displays the total number of documents stored in the `documents` variable. This is crucial for verifying that the document loading process (e.g., from a CSV or database) was successful and that the expected amount of data has been loaded into memory.


In [8]:
len(documents) # Calculates the length of the 'documents' list/variable, which represents the total count of loaded documents.


1000

In [9]:
print(documents[0].page_content)

Description: Ergonomic zero administration knowledge user


### Metadata Inspection

This cell inspects the metadata of the first document (`documents[0]`) loaded into memory. The `pp()` function (likely a custom print/pretty-print utility) displays this dictionary, allowing the user to verify crucial information such as source file names, page numbers, or chunk identifiers that were attached during the loading process.


In [13]:
pp(documents[100].metadata)


{'source': 'Computer / Network Security',
 'row': 100,
 'Website': 'https://www.chaney-eaton.com/',
 'Founded': '1993',
 'Number of employees': '3468'}
